# CodeBERT Smart-Contract Vulnerability Classifier — Corrected Training Notebook

**MSc Cybersecurity Capstone · revision run.** This notebook fine-tunes and evaluates the
CodeBERT function-level classifier with the methodological corrections requested in review.

### What changed from the previous version (read this)
1. **Focal loss corrected.** `p_t` is taken directly from `softmax(logits)` at the true class.
   The previous `pt = exp(-ce)` was only valid for plain cross-entropy; with class weighting
   and label smoothing it was a different quantity and distorted the rare-class focusing.
2. **Threshold selection made honest (no in-sample optimism).** Per-class thresholds are now
   chosen by **leave-one-fold-out** selection on out-of-fold probabilities and evaluated on the
   held-out fold. Both the unbiased (nested) and the old in-sample (upper-bound) numbers are
   reported side by side so the difference is explicit.
3. **Labelling procedure documented and function-level where available.** If the dataset carries
   function-level labels they are used directly; otherwise the contract-to-function propagation
   is applied and its label-noise exposure is quantified and printed.
4. **Leakage-free CV kept.** `StratifiedGroupKFold` groups whole contracts so none appears in
   both train and validation.
5. **Exact software versions printed** for the reproducibility appendix.

### How to run
- Kaggle: add `microsoft/codebert-base` (or let it download), set `DATA_CSV` to your corpus,
  enable GPU, then Run All. All tables (CSV) and figures (PNG) are written to the working dir
  for pasting into Chapter 4.
- The expected dataset schema is documented in the **Data** cell. A tiny synthetic fallback lets
  you confirm the pipeline runs before plugging in the real corpus.

In [ ]:
# =============================== CONFIG ===============================
from pathlib import Path

CONFIG = dict(
    MODEL_NAME     = "microsoft/codebert-base",
    DATA_CSV       = "/kaggle/input/smartbugs-functions/functions.csv",  # <-- EDIT to your corpus
    OUTPUT_DIR     = "/kaggle/working",
    EXPORT_DIR     = "/kaggle/working/codebert-smartbugs-final",

    MAX_LEN        = 256,
    N_SPLITS       = 5,
    EPOCHS         = 15,
    PATIENCE       = 3,            # early-stopping patience on val macro-F1
    BATCH_SIZE     = 8,
    GRAD_ACCUM     = 4,            # effective batch = 32
    LR             = 2e-5,
    WEIGHT_DECAY   = 0.01,
    WARMUP_RATIO   = 0.10,
    FOCAL_GAMMA    = 2.0,
    MSD_K          = 5,            # Multi-Sample Dropout passes
    DROPOUT        = 0.3,
    LABEL_SMOOTHING= 0.0,          # keep 0.0 for the clean focal loss (recommended)
    FP16           = True,
    SEED           = 42,

    # columns expected in DATA_CSV
    COL_SOURCE     = "source",        # the function source code (string)
    COL_LABEL      = "label",         # class name (string), e.g. 'benign','reentrancy',...
    COL_GROUP      = "contract_id",   # grouping key so a contract stays in one fold
    COL_FUNC_LABEL = "func_label",    # OPTIONAL function-level label; used if present
    BENIGN_NAME    = "benign",

    USE_SYNTHETIC_IF_MISSING = False, # if real CSV missing: STOP with an error (never train on toy data)
)
Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["EXPORT_DIR"]).mkdir(parents=True, exist_ok=True)
CONFIG

In [ ]:
# ===================== environment + reproducibility =====================
import os, random, json, sys, platform
import numpy as np
import torch

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG["SEED"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import transformers, sklearn, pandas as pd
VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "device": DEVICE,
}
print(json.dumps(VERSIONS, indent=2))
json.dump(VERSIONS, open(os.path.join(CONFIG["OUTPUT_DIR"], "environment.json"), "w"), indent=2)
print("\nPaste these exact versions into the reproducibility appendix (Fix 11).")

## Data: loading, labelling procedure and its validity

**Expected CSV schema** (one row per function):

| column | meaning |
|---|---|
| `source` | the function source code as a string |
| `label` | the contract-level vulnerability category (string) |
| `contract_id` | identifier of the parent contract (grouping key for leakage-free CV) |
| `func_label` *(optional)* | a **function-level** label if you have one; used instead of `label` when present |

**Labelling note (Fix 1).** If `func_label` is present it is used directly, giving true
function-level ground truth. If it is absent, each function inherits its parent contract's
`label` (contract-to-function propagation). That propagation is a known source of label noise:
a benign function inside a vulnerable contract receives a vulnerable label. The cell below
**quantifies** that exposure (functions per vulnerable contract) and prints it so the figure can
be cited in Chapter 3 and Section 5.6.

In [ ]:
# =============================== load data ===============================
import pandas as pd, numpy as np, os, glob

C = CONFIG
def make_synthetic():
    # tiny, structurally-valid toy corpus so the pipeline runs end to end.
    rng = np.random.default_rng(0)
    classes = ["benign","reentrancy","access_control","arithmetic","unchecked_calls","bad_randomness"]
    rows=[]
    cid=0
    for cls in classes:
        n_contracts = 12 if cls=="benign" else (3 if cls in ("bad_randomness",) else 6)
        for _ in range(n_contracts):
            cid+=1
            for f in range(rng.integers(2,5)):
                rows.append(dict(source=f"function f{f}() public {{ /* {cls} */ }}",
                                 label=cls, contract_id=f"C{cid}"))
    return pd.DataFrame(rows)

def _find_csv(configured):
    if os.path.exists(configured): return configured
    hits = sorted(glob.glob("/kaggle/input/**/functions.csv", recursive=True))
    return hits[0] if hits else None

resolved = _find_csv(C["DATA_CSV"])
IS_SYNTH = False
if resolved:
    df = pd.read_csv(resolved)
    print(f"Loaded {len(df)} rows from {resolved}")
elif C["USE_SYNTHETIC_IF_MISSING"]:
    df = make_synthetic(); IS_SYNTH = True
    print("="*72)
    print("WARNING: real dataset NOT found. Using SYNTHETIC toy data. RESULTS ARE NOT VALID.")
    print("="*72)
else:
    raise FileNotFoundError(
        f"functions.csv was not found at {C['DATA_CSV']!r} and not anywhere under /kaggle/input.\n"
        f"Fix: in the notebook's right panel click 'Add Input' and attach your dataset "
        f"(the one containing functions.csv), then Run All again.\n"
        f"Currently visible under /kaggle/input: {glob.glob('/kaggle/input/*')}")

# choose label column: function-level if available
if C["COL_FUNC_LABEL"] in df.columns and df[C["COL_FUNC_LABEL"]].notna().any():
    df["y_name"] = df[C["COL_FUNC_LABEL"]].fillna(df[C["COL_LABEL"]])
    LABELLING = "function-level (func_label column present)"
else:
    df["y_name"] = df[C["COL_LABEL"]]
    LABELLING = "contract-level propagation (no func_label column)"

df = df.rename(columns={C["COL_SOURCE"]:"source", C["COL_GROUP"]:"group"})
df = df[["source","y_name","group"]].dropna(subset=["source","y_name","group"]).reset_index(drop=True)

print("Labelling scheme:", LABELLING)

# --- quantify contract-to-function label-noise exposure (Fix 1 evidence) ---
fpc = df.groupby("group").size()
vuln_groups = df[df.y_name!=C["BENIGN_NAME"]].group.unique()
exposure = fpc[fpc.index.isin(vuln_groups)]
print(f"\nContracts: {df.group.nunique()} | functions: {len(df)}")
print(f"Vulnerable contracts: {len(vuln_groups)} | mean functions/vulnerable contract: {exposure.mean():.2f}")
print(f"=> up to {int((exposure.sum()-len(vuln_groups))) if len(vuln_groups) else 0} functions "
      f"could be mislabelled by propagation (non-annotated functions in vulnerable contracts).")

# ---- loud data check: refuse to proceed on the toy fallback or a wrong file ----
benign_share = 100*(df.y_name==C["BENIGN_NAME"]).mean()
print("\n" + "="*72)
print(f"DATA CHECK: {len(df)} functions | {df.y_name.nunique()} classes | "
      f"benign {benign_share:.1f}% | labelling: {LABELLING}")
print("="*72)
if not IS_SYNTH:
    assert len(df) >= 400, (f"Only {len(df)} rows loaded; the real corpus is ~993. "
                            f"Wrong file attached? Check the input path.")
    assert df.y_name.nunique() >= 8, (f"Only {df.y_name.nunique()} classes; expected ~11. "
                                      f"This looks like the toy data, not functions.csv.")
    print("OK: real dataset loaded. Proceed.")

In [ ]:
# ===================== class map + imbalance summary =====================
import matplotlib.pyplot as plt

classes = sorted(df.y_name.unique().tolist())
# keep benign first for readability if present
if CONFIG["BENIGN_NAME"] in classes:
    classes.remove(CONFIG["BENIGN_NAME"]); classes = [CONFIG["BENIGN_NAME"]]+classes
label2id = {c:i for i,c in enumerate(classes)}
id2label = {i:c for c,i in label2id.items()}
df["y"] = df.y_name.map(label2id)
NUM_LABELS = len(classes)

counts = df.y_name.value_counts().reindex(classes)
pct = (counts/len(df)*100).round(1)
summary = pd.DataFrame({"count":counts, "percent":pct})
summary.to_csv(os.path.join(CONFIG["OUTPUT_DIR"],"dataset_overview.csv"))
print(summary)

fig, ax = plt.subplots(figsize=(8,3.2))
ax.bar(range(NUM_LABELS), counts.values, color="#2a78d6")
ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(classes, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("functions"); ax.set_title("Dataset composition (function-level)")
for i,v in enumerate(counts.values): ax.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"],"fig1_dataset_overview.png"), dpi=200)
plt.show()

In [ ]:
# ===================== tokenizer + dataset =====================
import re, torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])

def normalize_code(src: str) -> str:
    src = re.sub(r"//.*?$|/\*.*?\*/", " ", str(src), flags=re.S|re.M)  # strip comments
    src = re.sub(r"\s+", " ", src).strip()                             # collapse whitespace
    return src

class FunctionDataset(Dataset):
    def __init__(self, sources, labels, tok, max_len):
        self.sources=list(sources); self.labels=list(labels); self.tok=tok; self.max_len=max_len
    def __len__(self): return len(self.sources)
    def __getitem__(self, i):
        enc = self.tok(normalize_code(self.sources[i]), truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[i], dtype=torch.long)}

In [ ]:
# ===================== model: CodeBERTClassifier (CLS pooling + Multi-Sample Dropout) =====================
import torch.nn as nn
from transformers import AutoModel

class CodeBERTClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.3, msd_k=5):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)
        self.msd_k = msd_k
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]                      # CLS-token pooling
        if self.training and self.msd_k and self.msd_k > 1:
            # Multi-Sample Dropout: average logits over K dropout masks (cheap in-pass ensemble)
            logits = torch.stack([self.classifier(self.dropout(cls)) for _ in range(self.msd_k)], 0).mean(0)
        else:
            logits = self.classifier(self.dropout(cls))        # dropout is identity in eval()
        return logits

In [ ]:
# ===================== corrected focal loss (Fix 2) =====================
import torch.nn.functional as F

class FocalLoss(nn.Module):
    # L_FL = - alpha_t (1 - p_t)^gamma log(p_t), with p_t = softmax(logits)[true class].
    # p_t is read directly from log_softmax, NOT as exp(-ce).
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0, reduction="mean"):
        super().__init__()
        self.alpha=alpha; self.gamma=float(gamma)
        self.ls=float(label_smoothing); self.reduction=reduction
    def forward(self, logits, targets):
        logp  = F.log_softmax(logits, dim=1)
        logpt = logp.gather(1, targets.view(-1,1)).squeeze(1)   # log p_t
        pt    = logpt.exp()                                     # p_t
        focal = (1.0 - pt).pow(self.gamma)
        if self.ls > 0.0:
            log_term = (1.0-self.ls)*(-logpt) + self.ls*(-logp.mean(dim=1))
        else:
            log_term = -logpt
        loss = focal * log_term
        if self.alpha is not None:
            loss = self.alpha.to(logits.device)[targets] * loss
        return loss.mean() if self.reduction=="mean" else (loss.sum() if self.reduction=="sum" else loss)

def inverse_frequency_alpha(y, num_labels, benign_id):
    counts = np.bincount(y, minlength=num_labels).astype(float)
    counts[counts==0] = 1.0
    w = counts.sum()/ (num_labels*counts)        # inverse frequency
    w = w / w[benign_id]                          # normalise so benign = 1.0
    return torch.tensor(w, dtype=torch.float)

In [ ]:
# ===================== train / evaluate one fold =====================
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score

def run_probs(model, loader):
    model.eval(); probs=[]; ys=[]
    with torch.no_grad():
        for b in loader:
            ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE)
            logits = model(ids, am)
            probs.append(torch.softmax(logits,1).cpu().numpy()); ys.append(b["labels"].numpy())
    return np.concatenate(probs), np.concatenate(ys)

def train_fold(tr_df, va_df, alpha):
    seed_everything(CONFIG["SEED"])
    model = CodeBERTClassifier(CONFIG["MODEL_NAME"], NUM_LABELS,
                               dropout=CONFIG["DROPOUT"], msd_k=CONFIG["MSD_K"]).to(DEVICE)
    tr = DataLoader(FunctionDataset(tr_df.source, tr_df.y, tokenizer, CONFIG["MAX_LEN"]),
                    batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
    va = DataLoader(FunctionDataset(va_df.source, va_df.y, tokenizer, CONFIG["MAX_LEN"]),
                    batch_size=CONFIG["BATCH_SIZE"])
    crit = FocalLoss(alpha=alpha, gamma=CONFIG["FOCAL_GAMMA"], label_smoothing=CONFIG["LABEL_SMOOTHING"])
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
    steps = (len(tr)//CONFIG["GRAD_ACCUM"]+1)*CONFIG["EPOCHS"]
    sched = get_cosine_schedule_with_warmup(opt, int(CONFIG["WARMUP_RATIO"]*steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=CONFIG["FP16"] and DEVICE=="cuda")

    best_f1=-1; best_state=None; patience=0; history=[]
    for ep in range(CONFIG["EPOCHS"]):
        model.train(); opt.zero_grad(); ep_loss=0
        for i,b in enumerate(tr):
            ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE); y=b["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CONFIG["FP16"] and DEVICE=="cuda"):
                logits=model(ids,am); loss=crit(logits,y)/CONFIG["GRAD_ACCUM"]
            scaler.scale(loss).backward(); ep_loss+=loss.item()*CONFIG["GRAD_ACCUM"]
            if (i+1)%CONFIG["GRAD_ACCUM"]==0:
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        vp,vy = run_probs(model, va)
        vf1 = f1_score(vy, vp.argmax(1), average="macro", zero_division=0)
        history.append({"epoch":ep,"train_loss":ep_loss/len(tr),"val_macro_f1":vf1})
        if vf1>best_f1:
            best_f1=vf1; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience=0
        else:
            patience+=1
            if patience>=CONFIG["PATIENCE"]: break
    model.load_state_dict(best_state)
    return model, history

In [ ]:
# ===================== leakage-free cross-validation (Fix 4) =====================
from sklearn.model_selection import StratifiedGroupKFold

alpha = inverse_frequency_alpha(df.y.values, NUM_LABELS, label2id[CONFIG["BENIGN_NAME"]])
print("Per-class alpha (benign=1):", {id2label[i]: round(float(a),2) for i,a in enumerate(alpha)})

sgkf = StratifiedGroupKFold(n_splits=CONFIG["N_SPLITS"], shuffle=True, random_state=CONFIG["SEED"])
oof_prob = np.zeros((len(df), NUM_LABELS))      # out-of-fold probabilities
oof_fold = np.full(len(df), -1)                 # which fold each row was tested in
fold_hist = {}; per_fold=[]

for fold,(tr_idx,va_idx) in enumerate(sgkf.split(df, df.y, groups=df.group)):
    print(f"\n===== Fold {fold}  train={len(tr_idx)} val={len(va_idx)} =====")
    model, hist = train_fold(df.iloc[tr_idx], df.iloc[va_idx], alpha)
    vp,vy = run_probs(model, DataLoader(
        FunctionDataset(df.iloc[va_idx].source, df.iloc[va_idx].y, tokenizer, CONFIG["MAX_LEN"]),
        batch_size=CONFIG["BATCH_SIZE"]))
    oof_prob[va_idx]=vp; oof_fold[va_idx]=fold; fold_hist[fold]=hist
    from sklearn.metrics import accuracy_score, matthews_corrcoef
    per_fold.append(dict(fold=fold,
        macro_f1=f1_score(vy,vp.argmax(1),average="macro",zero_division=0),
        weighted_f1=f1_score(vy,vp.argmax(1),average="weighted",zero_division=0),
        accuracy=accuracy_score(vy,vp.argmax(1)),
        mcc=matthews_corrcoef(vy,vp.argmax(1))))
    print(per_fold[-1])
    del model; torch.cuda.empty_cache()

per_fold_df = pd.DataFrame(per_fold)
per_fold_df.to_csv(os.path.join(CONFIG["OUTPUT_DIR"],"table_4_1_per_fold.csv"), index=False)
print("\nPer-fold:\n", per_fold_df)

In [ ]:
# ===================== aggregate CV metrics (Tables 4.1-4.3) =====================
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             average_precision_score, f1_score, accuracy_score, matthews_corrcoef)
mask = oof_fold>=0
y_true = df.y.values[mask]; P = oof_prob[mask]; y_pred = P.argmax(1)

summary = dict(
    weighted_f1 = f1_score(y_true,y_pred,average="weighted",zero_division=0),
    macro_f1    = f1_score(y_true,y_pred,average="macro",zero_division=0),
    accuracy    = accuracy_score(y_true,y_pred),
    mcc         = matthews_corrcoef(y_true,y_pred),
)
ci = per_fold_df[["macro_f1","weighted_f1","accuracy","mcc"]].agg(["mean","std"])
print("OOF aggregate:", {k:round(v,3) for k,v in summary.items()})
print("\nMean/std over folds:\n", ci.round(3))
ci.to_csv(os.path.join(CONFIG["OUTPUT_DIR"],"table_4_2_cv_summary.csv"))
json.dump({k:float(v) for k,v in summary.items()},
          open(os.path.join(CONFIG["OUTPUT_DIR"],"headline_metrics.json"),"w"), indent=2)

# per-class metrics + one-vs-rest AUC / AP
rows=[]
for c in range(NUM_LABELS):
    yc = (y_true==c).astype(int)
    rows.append(dict(cls=id2label[c], support=int(yc.sum()),
        precision=float((( (y_pred==c)&(y_true==c)).sum())/max((y_pred==c).sum(),1)),
        recall=float((((y_pred==c)&(y_true==c)).sum())/max(yc.sum(),1)),
        f1_default=f1_score((y_true==c).astype(int),(y_pred==c).astype(int),zero_division=0),
        roc_auc=(roc_auc_score(yc, P[:,c]) if 0<yc.sum()<len(yc) else float("nan")),
        pr_auc=(average_precision_score(yc, P[:,c]) if yc.sum()>0 else float("nan"))))
per_class_df=pd.DataFrame(rows)
per_class_df.to_csv(os.path.join(CONFIG["OUTPUT_DIR"],"table_4_3_per_class.csv"), index=False)
print("\nPer-class:\n", per_class_df.round(3))

## Threshold optimisation, done honestly (Fix 3)

Selecting a per-class threshold and then reporting its F1 on the *same* data is optimistic.
Here we report three things so the bias is explicit:

1. **Default (argmax / 0.5)** per-class F1, the baseline.
2. **In-sample optimised** per-class F1: thresholds chosen and evaluated on the pooled OOF
   predictions. This is the **optimistic upper bound** (what the previous version reported).
3. **Nested out-of-fold** per-class F1: for each fold, the threshold is chosen on the OOF
   predictions of the *other* folds and applied to this fold. This is the **unbiased** estimate
   to report as the headline in Chapter 4.

In [ ]:
# ===================== per-class thresholds: in-sample vs nested (Fix 3) =====================
def best_threshold(probs_c, y_c, grid=np.linspace(0.01,0.99,99)):
    best_t, best_f = 0.5, -1
    for t in grid:
        f = f1_score(y_c, (probs_c>=t).astype(int), zero_division=0)
        if f>best_f: best_f, best_t = f, t
    return best_t, best_f

rows=[]
for c in range(NUM_LABELS):
    yc = (y_true==c).astype(int); pc = P[:,c]
    # default
    f_def = f1_score(yc, (y_pred==c).astype(int), zero_division=0)
    # in-sample optimistic
    t_in, f_in = best_threshold(pc, yc)
    # nested out-of-fold: choose threshold on other folds, apply to this fold
    folds = oof_fold[mask]
    pred_nested = np.zeros_like(yc)
    for f in np.unique(folds):
        sel_other = folds!=f; sel_this = folds==f
        if yc[sel_other].sum()==0:          # class absent elsewhere -> fall back to 0.5
            t = 0.5
        else:
            t,_ = best_threshold(pc[sel_other], yc[sel_other])
        pred_nested[sel_this] = (pc[sel_this]>=t).astype(int)
    f_nested = f1_score(yc, pred_nested, zero_division=0)
    rows.append(dict(cls=id2label[c], support=int(yc.sum()),
                     f1_default=round(f_def,3), f1_insample_opt=round(f_in,3),
                     thr_insample=round(t_in,2), f1_nested_oof=round(f_nested,3)))
thr_df=pd.DataFrame(rows)
thr_df.to_csv(os.path.join(CONFIG["OUTPUT_DIR"],"table_4_4_thresholds.csv"), index=False)
print(thr_df)
print("\nReport f1_nested_oof as the honest per-class result. f1_insample_opt is the optimistic")
print("upper bound (what the previous version reported); quote it only as such.")

In [ ]:
# ===================== figures for Chapter 4 =====================
import matplotlib.pyplot as plt
OUT=CONFIG["OUTPUT_DIR"]

# training curves (per fold)
fig,ax=plt.subplots(1,2,figsize=(10,3.4))
for f,h in fold_hist.items():
    hh=pd.DataFrame(h)
    ax[0].plot(hh.epoch, hh.train_loss, label=f"fold {f}")
    ax[1].plot(hh.epoch, hh.val_macro_f1, label=f"fold {f}")
ax[0].set_title("Training focal loss"); ax[0].set_xlabel("epoch")
ax[1].set_title("Validation macro-F1"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_training_curves.png",dpi=200); plt.show()

# confusion matrix
cm=confusion_matrix(y_true,y_pred,labels=list(range(NUM_LABELS)))
fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(cm,cmap="Blues"); plt.colorbar(im)
ax.set_xticks(range(NUM_LABELS)); ax.set_xticklabels(classes,rotation=45,ha="right",fontsize=7)
ax.set_yticks(range(NUM_LABELS)); ax.set_yticklabels(classes,fontsize=7)
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j,i,cm[i,j],ha="center",va="center",fontsize=7,
                color="white" if cm[i,j]>cm.max()/2 else "black")
ax.set_title("Confusion matrix (OOF)"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_confusion_matrix.png",dpi=200); plt.show()

# threshold comparison bar
fig,ax=plt.subplots(figsize=(9,3.4)); x=np.arange(NUM_LABELS); w=0.27
ax.bar(x-w, thr_df.f1_default, w, label="default", color="#9aa6b2")
ax.bar(x,    thr_df.f1_insample_opt, w, label="in-sample (optimistic)", color="#eda100")
ax.bar(x+w, thr_df.f1_nested_oof, w, label="nested OOF (honest)", color="#1baf7a")
ax.set_xticks(x); ax.set_xticklabels(classes,rotation=45,ha="right",fontsize=7)
ax.set_ylabel("F1"); ax.set_title("Per-class F1: default vs in-sample vs nested thresholds"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig7_threshold_comparison.png",dpi=200); plt.show()

# calibration (ECE) over max-prob
def ece(probs,y_true,bins=10):
    conf=probs.max(1); pred=probs.argmax(1); acc=(pred==y_true).astype(float)
    e=0;
    for b in range(bins):
        lo,hi=b/bins,(b+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()>0: e+=abs(acc[m].mean()-conf[m].mean())*m.mean()
    return e
print("Expected Calibration Error (OOF):", round(ece(P,y_true),4))

In [ ]:
# ===================== PR curves, ROC curves, reliability diagram (Figures 4.4, 4.5, 4.7) =====================
from sklearn.metrics import precision_recall_curve, roc_curve, average_precision_score, roc_auc_score
OUT=CONFIG["OUTPUT_DIR"]
PALETTE=['#2a78d6','#eb6834','#1baf7a','#eda100','#e87ba4','#4a3aa7','#e34948','#7d5ba6','#12386b','#8c7b6b','#5b6472']

# ---- Figure 4.4: per-class precision-recall curves ----
plt.figure(figsize=(7,5))
for c in range(NUM_LABELS):
    yc=(y_true==c).astype(int)
    if yc.sum()==0: continue
    pr,rc,_=precision_recall_curve(yc,P[:,c]); ap=average_precision_score(yc,P[:,c])
    plt.plot(rc,pr,color=PALETTE[c%len(PALETTE)],label=f"{id2label[c]} (AP={ap:.2f})")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Per-class precision-recall curves (out-of-fold)")
plt.legend(fontsize=7,loc="upper right"); plt.tight_layout()
plt.savefig(f"{OUT}/fig5_pr_curves.png",dpi=200); plt.show()

# ---- Figure 4.5: per-class ROC curves ----
plt.figure(figsize=(7,5))
for c in range(NUM_LABELS):
    yc=(y_true==c).astype(int)
    if 0<yc.sum()<len(yc):
        fpr,tpr,_=roc_curve(yc,P[:,c]); auc=roc_auc_score(yc,P[:,c])
        plt.plot(fpr,tpr,color=PALETTE[c%len(PALETTE)],label=f"{id2label[c]} (AUC={auc:.2f})")
plt.plot([0,1],[0,1],'k--',lw=0.8)
plt.xlabel("false positive rate"); plt.ylabel("true positive rate"); plt.title("Per-class ROC curves (out-of-fold)")
plt.legend(fontsize=7,loc="lower right"); plt.tight_layout()
plt.savefig(f"{OUT}/fig6_roc_curves.png",dpi=200); plt.show()

# ---- Figure 4.7: reliability diagram + ECE (overall and per well-supported class) ----
def reliability(probs, y_true, bins=10):
    conf=probs.max(1); pred=probs.argmax(1); acc=(pred==y_true).astype(float)
    xs=[]; ys=[]; e=0.0
    for b in range(bins):
        lo,hi=b/bins,(b+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()>0:
            xs.append(conf[m].mean()); ys.append(acc[m].mean()); e+=abs(acc[m].mean()-conf[m].mean())*m.mean()
    return xs,ys,e
xs,ys,ECE=reliability(P,y_true)
plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],'k--',lw=0.8,label="perfect calibration")
plt.plot(xs,ys,'o-',color='#2a78d6',label=f"model (ECE={ECE:.3f})")
plt.xlabel("mean predicted confidence"); plt.ylabel("empirical accuracy")
plt.title("Reliability diagram (out-of-fold)"); plt.legend(fontsize=9); plt.tight_layout()
plt.savefig(f"{OUT}/fig8_calibration.png",dpi=200); plt.show()

print("\n" + "="*60)
print(f"EXPECTED CALIBRATION ERROR (OOF) = {ECE:.4f}")
print("Paste this value into Section 4.7 in place of [INSERT ECE FROM CORRECTED RUN].")
print("="*60)

In [ ]:
# ===================== final fit on ALL data + export (integration-compatible) =====================
seed_everything(CONFIG["SEED"])
final = CodeBERTClassifier(CONFIG["MODEL_NAME"], NUM_LABELS,
                           dropout=CONFIG["DROPOUT"], msd_k=CONFIG["MSD_K"]).to(DEVICE)
full = DataLoader(FunctionDataset(df.source, df.y, tokenizer, CONFIG["MAX_LEN"]),
                  batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
crit = FocalLoss(alpha=alpha, gamma=CONFIG["FOCAL_GAMMA"], label_smoothing=CONFIG["LABEL_SMOOTHING"])
opt = torch.optim.AdamW(final.parameters(), lr=CONFIG["LR"], weight_decay=CONFIG["WEIGHT_DECAY"])
steps=(len(full)//CONFIG["GRAD_ACCUM"]+1)*CONFIG["EPOCHS"]
sched=get_cosine_schedule_with_warmup(opt,int(CONFIG["WARMUP_RATIO"]*steps),steps)
scaler=torch.cuda.amp.GradScaler(enabled=CONFIG["FP16"] and DEVICE=="cuda")
for ep in range(CONFIG["EPOCHS"]):
    final.train(); opt.zero_grad()
    for i,b in enumerate(full):
        ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE); y=b["labels"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CONFIG["FP16"] and DEVICE=="cuda"):
            loss=crit(final(ids,am),y)/CONFIG["GRAD_ACCUM"]
        scaler.scale(loss).backward()
        if (i+1)%CONFIG["GRAD_ACCUM"]==0:
            scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()

ED=CONFIG["EXPORT_DIR"]
final.backbone.save_pretrained(ED)                    # encoder (backbone.*) for local loading
tokenizer.save_pretrained(ED)
torch.save(final.classifier.state_dict(), os.path.join(ED,"classifier_head.pt"))
torch.save(final.state_dict(),            os.path.join(ED,"full_model.pt"))
meta = dict(architecture="CodeBERTClassifier", pooling="cls", num_labels=NUM_LABELS,
            label2id=label2id, id2label={int(k):v for k,v in id2label.items()},
            max_len=CONFIG["MAX_LEN"], gamma=CONFIG["FOCAL_GAMMA"],
            alpha={id2label[i]:float(a) for i,a in enumerate(alpha)},
            thresholds_nested={r["cls"]:float(r["f1_nested_oof"]) for _,r in thr_df.iterrows()},
            versions=VERSIONS)
json.dump(meta, open(os.path.join(ED,"model_metadata.json"),"w"), indent=2)
print("Exported to", ED, "->", os.listdir(ED))
print("These files match the nlp_service analyzer (backbone + classifier_head.pt / full_model.pt).")

## Updating the dissertation from this run

All outputs are in the working directory:

- `table_4_1_per_fold.csv`, `table_4_2_cv_summary.csv`, `table_4_3_per_class.csv`,
  `table_4_4_thresholds.csv` → paste into Chapter 4 Tables 4.1 to 4.4.
- `headline_metrics.json` → the Abstract and Chapter 5 headline numbers.
- `fig1_dataset_overview.png`, `fig2_training_curves.png`, `fig4_confusion_matrix.png`,
  `fig7_threshold_comparison.png` → Chapter 4 figures.
- `environment.json` → the reproducibility appendix (exact versions, Fix 11).
- `codebert-smartbugs-final/` → the exported model, compatible with the Django `nlp_service`.

**Report the corrected numbers as they come out.** In Chapter 4 state that the earlier run used
a mis-specified focal loss and an in-sample threshold estimate, and that these results use the
corrected loss and nested out-of-fold thresholds. That before/after honesty is exactly what the
review asked for.